In [ ]:
import numpy as np
import pandas as pd

import effiara
from effiara.annotator_reliability import Annotations
from effiara.label_generators import LabelGenerator


def parse_labels(x):
    if pd.isna(x) or str(x).strip() == "":
        return set()
    return {p.strip() for p in str(x).split(",") if p.strip()}


def jaccard(a, b):
    a, b = parse_labels(a), parse_labels(b)
    if not a and not b:
        return np.nan
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)


class MultiLabelGenerator(LabelGenerator):
    @classmethod
    def from_annotations(cls, df, suffix="_label"):
        annotators = sorted({
            c[:-len(suffix)]
            for c in df.columns
            if c.endswith(suffix) and not c.startswith("re_")
        })

        labels = sorted({
            lab
            for c in df.columns
            if c.endswith(suffix)
            for x in df[c].dropna()
            for lab in parse_labels(x)
        })

        label_mapping = {lab: i for i, lab in enumerate(labels)}
        return cls(
            annotators=annotators,
            label_mapping=label_mapping,
            label_suffixes=[suffix],
        )

    def _to_multihot(self, x):
        vec = np.zeros(len(self.label_mapping), dtype=float)
        for lab in parse_labels(x):
            if lab in self.label_mapping:
                vec[self.label_mapping[lab]] = 1.0
        return vec

    def add_annotation_prob_labels(self, df):
        df = df.copy()
        for annotator in self.annotators:
            col = f"{annotator}_label"
            df[f"{annotator}_prob"] = df[col].apply(self._to_multihot)
        return df

    def add_sample_prob_labels(self, df, reliability_dict=None):
        df = df.copy()
        if not any(c.endswith("_prob") for c in df.columns):
            df = self.add_annotation_prob_labels(df)

        def aggregate(row):
            vectors = []
            weights = []

            for annotator in self.annotators:
                lab = row.get(f"{annotator}_label")
                if parse_labels(lab):
                    vectors.append(row[f"{annotator}_prob"])
                    weights.append(
                        1.0 if reliability_dict is None
                        else reliability_dict.get(annotator, 1.0)
                    )

            if not vectors:
                return np.zeros(len(self.label_mapping), dtype=float)

            return np.average(np.vstack(vectors), axis=0, weights=np.array(weights))

        df["consensus_prob"] = df.apply(aggregate, axis=1)
        return df

    def add_sample_hard_labels(self, df, threshold=0.5):
        df = df.copy()
        if "consensus_prob" not in df.columns:
            df = self.add_sample_prob_labels(df)

        inv = {i: lab for lab, i in self.label_mapping.items()}

        def hard_labels(prob):
            return sorted(inv[i] for i, p in enumerate(prob) if p >= threshold)

        df["consensus_hard"] = df["consensus_prob"].apply(hard_labels)
        return df


class MultiLabelAnnotations(Annotations):
    def calculate_inter_annotator_agreement(self):
        scores = {}

        for i, a in enumerate(self.annotators):
            for b in self.annotators[i + 1:]:
                a_col = f"{a}{self.agreement_suffix}"
                b_col = f"{b}{self.agreement_suffix}"

                vals = [
                    jaccard(x, y)
                    for x, y in zip(self.df[a_col], self.df[b_col])
                    if parse_labels(x) and parse_labels(y)
                ]

                if len(vals) >= self.overlap_threshold:
                    score = float(np.nanmean(vals))
                    scores[(a, b)] = score
                    self.G.add_edge(a, b, agreement=score)

        self.avg_inter_annotator_agreement = (
            float(np.mean(list(scores.values()))) if scores else np.nan
        )

    def calculate_intra_annotator_agreement(self):
        for user in self.annotators:
            col = f"{user}{self.agreement_suffix}"
            re_col = f"re_{user}{self.agreement_suffix}"

            if self.reannotations and re_col in self.df.columns:
                vals = [
                    jaccard(x, y)
                    for x, y in zip(self.df[col], self.df[re_col])
                    if parse_labels(x) and parse_labels(y)
                ]
                self.G.nodes[user]["intra_agreement"] = (
                    float(np.nanmean(vals)) if vals else 1.0
                )
            else:
                self.G.nodes[user]["intra_agreement"] = 1.0

In [ ]:
import pandas as pd

annotations = pd.read_csv("annotated/annotations-aggregated.csv")

label_generator = MultiLabelGenerator.from_annotations(annotations)

annos = MultiLabelAnnotations(
    annotations,
    label_generator=label_generator,
    reannotations=True,
    overlap_threshold=1,
)

print(annos)

#annos.generate_final_labels_and_sample_weights()
print(annos.df[["sample_id", "consensus_prob", "consensus_hard"]])

Node A0 has the following attributes:
  reliability: 0.9474149556911347
  intra_agreement: 0.8125
  avg_inter_agreement: 0.7160552759305733

Node A1 has the following attributes:
  reliability: 1.0292137973327233
  intra_agreement: 1.0
  avg_inter_agreement: 0.696212653636063

Node A2 has the following attributes:
  reliability: 1.0394696828046939
  intra_agreement: 0.8875
  avg_inter_agreement: 0.82787624322508

Node A3 has the following attributes:
  reliability: 1.0435630471699877
  intra_agreement: 0.85
  avg_inter_agreement: 0.8729216137640715

Node A4 has the following attributes:
  reliability: 0.9403385170014602
  intra_agreement: 0.7875000000000001
  avg_inter_agreement: 0.7251595978556494


    sample_id                                     consensus_prob  \
0           0  [0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...   
1           1  [0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...   
2           2  [0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...   
3           3  [0.